## Проект: Retrieval-Augmented Generation (RAG) на данных StackOverflow
## Описание проекта

Цель проекта — построить систему поиска и генерации ответов на основе данных StackOverflow с использованием подхода Retrieval-Augmented Generation (RAG).

В рамках проекта реализован полный пайплайн подготовки данных, разбиения документов на чанки, генерации эмбеддингов и хранения их в векторной базе данных PostgreSQL с расширением pgvector.

Подход RAG позволяет находить релевантные фрагменты текста с помощью векторного поиска и использовать их в качестве контекста для генерации ответа языковой моделью.

In [1]:
from src.ingestion.loader import DataLoader
from src.core.config import Settings
from src.pipeline import Pipeline 
import psycopg2
from src.ingestion.chunker import Chunker
from src.indexing.embedder import EmbeddingGenerator
from src.ingestion.cleaner import Preprocessing
from src.indexing.vector_store import VectorStore
from nltk.tokenize import word_tokenize
from src.core.logging import setup_logging
from pathlib import Path
from src.core.queries import INSERT_CHUNK_QUERY, INSERT_CHUNK_QUERY_TEST

import logging
settings = Settings()

setup_logging(settings)

logger = logging.getLogger(__name__)
logger.info("Логгер запущен")


2026-03-20 10:04:00,556 | INFO | __main__ | Логгер запущен


In [2]:
settings = Settings()
pipeline = Pipeline(settings)

data = pipeline.run()

with psycopg2.connect(**settings.DB_PARAMS) as conn:
    with conn.cursor() as cursor:
        rows = pipeline.db_saver.build_insert_rows(data)
        pipeline.db_saver.insert_rows(rows, conn, cursor, INSERT_CHUNK_QUERY_TEST)

2026-03-20 10:04:06,736 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-20 10:04:08,191 | INFO | src.pipeline | Шаг 1: Загрузка данных
2026-03-20 10:04:08,202 | INFO | src.ingestion.loader | Данные из файла data/raw/Questions.csv успешно загружены, 200 строк.
2026-03-20 10:04:08,211 | INFO | src.ingestion.loader | Данные из файла data/raw/Answers.csv успешно загружены, 200 строк.
2026-03-20 10:04:08,217 | INFO | src.ingestion.loader | Данные из файла data/raw/Tags.csv успешно загружены, 200 строк.
2026-03-20 10:04:08,219 | INFO | src.core.config | Время выполнения функции 'load_data': 0.0267 секунд
2026-03-20 10:04:08,220 | INFO | src.pipeline | questions=200, answers=200
2026-03-20 10:04:08,221 | INFO | src.pipeline | Шаг 2: Предобработ

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

2026-03-20 10:04:08,634 | INFO | src.pipeline | Pipeline завершён
2026-03-20 10:04:08,634 | INFO | src.core.config | Время выполнения функции 'run': 0.4439 секунд
2026-03-20 10:04:08,750 | INFO | src.indexing.vector_store | Успешно вставлено 76 чанков из 76.


### 6. Поиск

In [8]:
settings = Settings()
embedder = EmbeddingGenerator()
loader_data = DataLoader(settings)
vector = VectorStore(settings)

query = 'SELECT'
query_embedding = embedder.encode(query)

with psycopg2.connect(**settings.DB_PARAMS) as conn:
    with conn.cursor() as cursor:
        doc = vector.select_from_db(query_embedding, conn, cursor)

2026-03-20 09:51:24,425 | INFO | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2026-03-20 09:51:24,656 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-20 09:51:24,683 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-03-20 09:51:24,860 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-20 09:51:24,886 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-20 09:51:25,040 |

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-20 09:51:26,382 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-20 09:51:26,410 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
2026-03-20 09:51:26,563 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-20 09:51:26,592 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolv

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [4]:
display(doc)

[{'chunk_id': 'q16412050_a16412083_c1',
  'chunk_text': 'SELECT *',
  'similarity': 0.08965897025710357,
  'question_id': 16412050,
  'answer_id': 16412083,
  'chunk_index': 1,
  'title': 'select only specific fields in joined table',
  'tags': 'mysql, php, sql'},
 {'chunk_id': 'q33159370_a33160144_c3',
  'chunk_text': 'option in select',
  'similarity': 0.13084352016448975,
  'question_id': 33159370,
  'answer_id': 33160144,
  'chunk_index': 3,
  'title': 'Error with Select (Angular JS) -> It shows blank option not the real first option (look screenshots)',
  'tags': 'angularjs, css, html, twitter-bootstrap'},
 {'chunk_id': 'q32834890_a32834958_c2',
  'chunk_text': 'SELECT ...',
  'similarity': 0.14032459259033203,
  'question_id': 32834890,
  'answer_id': 32834958,
  'chunk_index': 2,
  'title': 'Alter view from columns in the View',
  'tags': 'oracle, sql, view'},
 {'chunk_id': 'q8991940_a8992142_c2',
  'chunk_text': 'a select ?',
  'similarity': 0.17342465082259917,
  'question_id'